In [1]:
import pandas as pd
import json
import matplotlib.pyplot as plt

In [3]:
#dataset bursty
experiments = [
    {"id": "exp_20250625_014444", "name": "Openwhisk"},
    {"id": "exp_20250625_071954", "name": "Proposed"},
    {"id": "exp_20250624_210136", "name": "Histogram"},
    {"id": "exp_20250624_165438", "name": "Pagurus"},
]
#Normal
# experiments = [
#     {"id": "exp_20250626_205804", "name": "Openwhisk"},
#     {"id": "exp_20250627_042542", "name": "NMIG"},
#     {"id": "exp_20250626_155721", "name": "Histogram"},
#     {"id": "exp_20250626_113035", "name": "Pagurus"},
# ]


# # Similar

# experiments = [
#     {"id": "exp_20250625_162425", "name": "Openwhisk"},
#     {"id": "exp_20250625_113106", "name": "NMIG"},
#     {"id": "exp_20250625_211541", "name": "Histogram"},
#     {"id": "exp_20250626_050332", "name": "Pagurus"},
# ]


mem_cost_per_sec_per_mb = 0.00001
cpu_cost_per_sec= 0.00005
memory_mb = 1024

In [4]:
# Placeholder for plotting data
plot_data = {}

# Iterate over all experiments
for exp in experiments:
    # Load your dataframe here. Replace with actual loading logic:
    path = f"../results/{exp['id']}/docker_log.csv"
    df = pd.read_csv(path)
    df = pd.read_csv(path,names=['Timestamp','name','image','status'])

    start_time = df['Timestamp'].min()
    df['ElapsedTime'] = (df['Timestamp'] - start_time) / 1000  # in seconds
    
    
    # Process timestamps
    timestamps = sorted(df['Timestamp'].unique())

    # Build container status per timestamp
    container_status = {}
    for name, group in df.groupby('name'):
        group = group.sort_values('Timestamp')
        status_list = []
        last_status = None
        idx = 0
        for t in timestamps:
            while idx < len(group) and group.iloc[idx]['Timestamp'] <= t:
                last_status = group.iloc[idx]['status']
                idx += 1
            status_list.append(last_status)
        container_status[name] = status_list

    # Calculate cumulative cost
    cumulative_costs = []
    total_cost = 0.0
    for i in range(1, len(timestamps)):
        interval_sec = (timestamps[i] - timestamps[i-1]) / 1000.0
        for name, status_list in container_status.items():
            status = status_list[i-1]
            if status is None:
                continue
            mem_cost = interval_sec * mem_cost_per_sec_per_mb * memory_mb
            if status == 'running':
                cpu_cost = interval_sec * cpu_cost_per_sec
                total_cost += mem_cost + cpu_cost
            elif status == 'paused':
                total_cost += mem_cost
        cumulative_costs.append(total_cost)

    elapsed_secs = [(t - timestamps[0]) / 1000 for t in timestamps[1:]]
    plot_data[exp['name']] = (elapsed_secs, cumulative_costs)


In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

# -----------------------------
# Cost parameters
# -----------------------------
mem_cost_per_sec_per_mb = 0.00001
cpu_cost_per_sec = 0.00005
memory_mb = 1024

MEM_RATE = mem_cost_per_sec_per_mb * memory_mb
CPU_RATE = cpu_cost_per_sec

BASE_DIR = "../results"

# -----------------------------
# Experiments for all datasets
# -----------------------------
datasets = {
    "Bursty": [
        {"id": "exp_20250625_014444", "name": "OpenWhisk"},
        {"id": "exp_20250625_071954", "name": "NMIG"},
        {"id": "exp_20250624_210136", "name": "Histogram"},
        {"id": "exp_20250624_165438", "name": "Pagurus"},
    ],
    "Normal": [
        {"id": "exp_20250626_205804", "name": "OpenWhisk"},
        {"id": "exp_20250627_042542", "name": "NMIG"},
        {"id": "exp_20250626_155721", "name": "Histogram"},
        {"id": "exp_20250626_113035", "name": "Pagurus"},
    ],
    "Similar": [
        {"id": "exp_20250625_162425", "name": "OpenWhisk"},
        {"id": "exp_20250625_113106", "name": "NMIG"},
        {"id": "exp_20250625_211541", "name": "Histogram"},
        {"id": "exp_20250626_050332", "name": "Pagurus"},
    ],
}


# -----------------------------
# Load log safely
# -----------------------------
def load_docker_log(path):
    df = pd.read_csv(
        path,
        header=None,
        names=["Timestamp", "name", "image", "status"]
    )

    df["Timestamp"] = pd.to_numeric(df["Timestamp"], errors="coerce")
    df = df.dropna(subset=["Timestamp", "name", "status"])
    df["Timestamp"] = df["Timestamp"].astype("int64")

    return df


# -----------------------------
# Original method, your current technique
# -----------------------------
def compute_cost_original(path):
    df = load_docker_log(path)

    timestamps = sorted(df["Timestamp"].unique())

    container_status = {}

    for name, group in df.groupby("name"):
        group = group.sort_values("Timestamp")
        status_list = []
        last_status = None
        idx = 0

        for t in timestamps:
            while idx < len(group) and group.iloc[idx]["Timestamp"] <= t:
                last_status = group.iloc[idx]["status"]
                idx += 1
            status_list.append(last_status)

        container_status[name] = status_list

    total_cost = 0.0

    for i in range(1, len(timestamps)):
        interval_sec = (timestamps[i] - timestamps[i - 1]) / 1000.0

        for name, status_list in container_status.items():
            status = status_list[i - 1]

            if status is None:
                continue

            mem_cost = interval_sec * MEM_RATE

            if status == "running":
                cpu_cost = interval_sec * CPU_RATE
                total_cost += mem_cost + cpu_cost

            elif status == "paused":
                total_cost += mem_cost

    return total_cost


# -----------------------------
# Faster interval method
# -----------------------------
def compute_cost_fast(path):
    df = load_docker_log(path)

    df = df.sort_values(["name", "Timestamp"])

    # Avoid duplicate status records at the same timestamp for same container
    df = df.drop_duplicates(subset=["name", "Timestamp"], keep="last")

    global_end_time = df["Timestamp"].max()

    df["next_timestamp"] = df.groupby("name")["Timestamp"].shift(-1)
    df["next_timestamp"] = df["next_timestamp"].fillna(global_end_time)

    df["duration_sec"] = (df["next_timestamp"] - df["Timestamp"]) / 1000.0
    df["duration_sec"] = df["duration_sec"].clip(lower=0)

    active_memory_statuses = ["running", "paused"]
    active_cpu_statuses = ["running"]

    memory_cost = df.loc[
        df["status"].isin(active_memory_statuses),
        "duration_sec"
    ].sum() * MEM_RATE

    cpu_cost = df.loc[
        df["status"].isin(active_cpu_statuses),
        "duration_sec"
    ].sum() * CPU_RATE

    total_cost = memory_cost + cpu_cost

    return total_cost


# -----------------------------
# Compare all experiments
# -----------------------------
def verify_all_experiments(tolerance=1e-6):
    rows = []

    for dataset_name, experiments in datasets.items():
        for exp in experiments:
            path = Path(BASE_DIR) / exp["id"] / "docker_log.csv"

            old_cost = compute_cost_original(path)
            fast_cost = compute_cost_fast(path)

            diff = abs(old_cost - fast_cost)

            if old_cost != 0:
                diff_percent = (diff / old_cost) * 100
            else:
                diff_percent = 0.0

            status = "MATCH" if diff <= tolerance else "CHECK"

            rows.append({
                "Dataset": dataset_name,
                "Method": exp["name"],
                "Experiment ID": exp["id"],
                "Original Cost": old_cost,
                "Fast Cost": fast_cost,
                "Difference": diff,
                "Difference (%)": diff_percent,
                "Status": status
            })

    result_df = pd.DataFrame(rows)

    return result_df


verification_df = verify_all_experiments()

print("\nVerification Result:")
print(verification_df.to_string(index=False))

# Save result
verification_df.to_csv("cpu_mem_cost_verification.csv", index=False)

# Show only experiments that do not match
problem_df = verification_df[verification_df["Status"] == "CHECK"]

print("\nExperiments needing checking:")
if len(problem_df) == 0:
    print("All experiments match. The faster method is safe to use.")
else:
    print(problem_df.to_string(index=False))

KeyboardInterrupt: 


Detailed CPU + Memory Cost
Dataset    Method       Experiment ID  Memory Cost  CPU Cost  CPU+Mem Cost
 Normal OpenWhisk exp_20250626_205804  11296.65536  18.46015   11315.11551
 Normal      NMIG exp_20250627_042542   8761.29280   2.87390    8764.16670
 Normal Histogram exp_20250626_155721  49165.59872  23.97120   49189.56992
 Normal   Pagurus exp_20250626_113035   9514.96704  12.62815    9527.59519
Similar OpenWhisk exp_20250625_162425   2382.06976   3.17775    2385.24751
Similar      NMIG exp_20250625_113106   3347.86560   4.49450    3352.36010
Similar Histogram exp_20250625_211541  37111.04000  26.56270   37137.60270
Similar   Pagurus exp_20250626_050332   2779.10528   3.40745    2782.51273
 Bursty OpenWhisk exp_20250625_014444  13689.09824  18.01685   13707.11509
 Bursty      NMIG exp_20250625_071954  14161.15200   2.55545   14163.70745
 Bursty Histogram exp_20250624_210136  48565.18656  30.33835   48595.52491
 Bursty   Pagurus exp_20250624_165438  14934.21056  15.10585   14949.316

In [8]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# 1. Cost parameters
# ============================================================

mem_cost_per_sec_per_mb = 0.00001
cpu_cost_per_sec = 0.00005
memory_mb = 1024

MEM_RATE = mem_cost_per_sec_per_mb * memory_mb
CPU_RATE = cpu_cost_per_sec

BASE_DIR = Path("../results")


# ============================================================
# 2. Experiment list
# ============================================================

datasets = {
    "Normal": [
        {"id": "exp_20250626_205804", "name": "OpenWhisk"},
        {"id": "exp_20250627_042542", "name": "NMIG"},
        {"id": "exp_20250626_155721", "name": "Histogram"},
        {"id": "exp_20250626_113035", "name": "Pagurus"},
    ],

    "Similar": [
        {"id": "exp_20250625_162425", "name": "OpenWhisk"},
        {"id": "exp_20250625_113106", "name": "NMIG"},
        {"id": "exp_20250625_211541", "name": "Histogram"},
        {"id": "exp_20250626_050332", "name": "Pagurus"},
    ],

    "Bursty": [
        {"id": "exp_20250625_014444", "name": "OpenWhisk"},
        {"id": "exp_20250625_071954", "name": "NMIG"},
        {"id": "exp_20250624_210136", "name": "Histogram"},
        {"id": "exp_20250624_165438", "name": "Pagurus"},
    ],
}


# ============================================================
# 3. Candidate request/result log names
#    Add your real filename here if it is different.
# ============================================================

CANDIDATE_COMPLETION_FILES = [
    "invocation_log.csv",
    "request_log.csv",
    "requests.csv",
    "results.csv",
    "result.csv",
    "latency_log.csv",
    "execution_log.csv",
    "response_log.csv",
    "metrics.csv",
]

# Possible columns that may contain request completion time
CANDIDATE_END_COLUMNS = [
    "end_time",
    "endTime",
    "finish_time",
    "finished_time",
    "completion_time",
    "completed_time",
    "response_time",
    "end_timestamp",
    "finish_timestamp",
    "completed_timestamp",
    "completion_timestamp",
    "EndTime",
    "FinishTime",
    "CompletedTime",
]

# Possible columns that may contain request start or arrival time
CANDIDATE_START_COLUMNS = [
    "start_time",
    "startTime",
    "arrival_time",
    "invoke_time",
    "submit_time",
    "request_time",
    "start_timestamp",
    "arrival_timestamp",
    "invoke_timestamp",
    "StartTime",
    "ArrivalTime",
]


# ============================================================
# 4. Helper functions
# ============================================================

def convert_time_series_to_ms(series):
    """
    Converts a numeric timestamp column to milliseconds if it appears to be in seconds.
    Assumes Docker log timestamps are in milliseconds.
    """
    s = pd.to_numeric(series, errors="coerce").dropna()

    if len(s) == 0:
        return s

    median_value = s.median()

    # Unix timestamp in seconds, around 1.7e9 for year 2025
    if 1e9 <= median_value < 1e11:
        s = s * 1000

    return s.astype("int64")


def load_docker_log(docker_log_path):
    """
    Loads docker_log.csv with columns:
    Timestamp, name, image, status
    """
    df = pd.read_csv(
        docker_log_path,
        header=None,
        names=["Timestamp", "name", "image", "status"],
        usecols=["Timestamp", "name", "status"]
    )

    df["Timestamp"] = pd.to_numeric(df["Timestamp"], errors="coerce")
    df = df.dropna(subset=["Timestamp", "name", "status"])

    df["Timestamp"] = df["Timestamp"].astype("int64")
    df["name"] = df["name"].astype(str).str.strip()
    df["status"] = df["status"].astype(str).str.strip().str.lower()

    df = df.sort_values(["name", "Timestamp"])
    df = df.drop_duplicates(subset=["name", "Timestamp"], keep="last")

    return df


def find_completion_window(exp_dir, docker_df):
    """
    Finds workload start and end time from request or invocation logs.

    If no request completion file is found, it falls back to Docker log start and end.
    This fallback is safe to run, but may overcount if Docker monitoring continues
    after the workload has already finished.
    """
    docker_start = int(docker_df["Timestamp"].min())
    docker_end = int(docker_df["Timestamp"].max())

    selected_file = None
    selected_start = None
    selected_end = None

    for filename in CANDIDATE_COMPLETION_FILES:
        path = exp_dir / filename

        if not path.exists():
            continue

        try:
            temp = pd.read_csv(path)
        except Exception:
            continue

        if temp.empty:
            continue

        # Find end time column
        end_col = None
        for col in CANDIDATE_END_COLUMNS:
            if col in temp.columns:
                end_col = col
                break

        if end_col is None:
            continue

        end_series = convert_time_series_to_ms(temp[end_col])

        if len(end_series) == 0:
            continue

        candidate_end = int(end_series.max())

        # Find start time column if available
        start_col = None
        for col in CANDIDATE_START_COLUMNS:
            if col in temp.columns:
                start_col = col
                break

        if start_col is not None:
            start_series = convert_time_series_to_ms(temp[start_col])
            if len(start_series) > 0:
                candidate_start = int(start_series.min())
            else:
                candidate_start = docker_start
        else:
            candidate_start = docker_start

        # Use this candidate only if it overlaps with Docker monitoring time.
        # This avoids using unrelated relative timestamps.
        margin_ms = 10 * 60 * 1000  # 10 minutes margin

        if candidate_end < docker_start - margin_ms:
            continue

        if candidate_start > docker_end + margin_ms:
            continue

        selected_file = filename
        selected_start = max(candidate_start, docker_start)
        selected_end = min(candidate_end, docker_end)

        break

    if selected_end is None:
        selected_file = "docker_log.csv fallback"
        selected_start = docker_start
        selected_end = docker_end

    if selected_end <= selected_start:
        selected_start = docker_start
        selected_end = docker_end
        selected_file = "docker_log.csv fallback due to invalid window"

    return {
        "start_ms": int(selected_start),
        "end_ms": int(selected_end),
        "source": selected_file,
        "docker_start_ms": docker_start,
        "docker_end_ms": docker_end,
    }


def prepare_events_for_window(df, window_start_ms, window_end_ms):
    """
    Builds event intervals within the real experiment window.

    Important:
    If a container entered running or paused before the window start,
    this function carries that last known status into the window.
    """
    rows = []

    for container_name, group in df.groupby("name"):
        group = group.sort_values("Timestamp")

        before_or_at_start = group[group["Timestamp"] <= window_start_ms]
        inside_window = group[
            (group["Timestamp"] > window_start_ms) &
            (group["Timestamp"] <= window_end_ms)
        ]

        # Carry the last known status into the window start
        if len(before_or_at_start) > 0:
            last_row = before_or_at_start.iloc[-1].copy()
            last_row["Timestamp"] = window_start_ms
            rows.append(last_row)

        # Keep status changes inside the window
        if len(inside_window) > 0:
            rows.extend([row for _, row in inside_window.iterrows()])

    if len(rows) == 0:
        return pd.DataFrame(columns=["Timestamp", "name", "status"])

    event_df = pd.DataFrame(rows)
    event_df = event_df[["Timestamp", "name", "status"]]
    event_df = event_df.sort_values(["name", "Timestamp"])
    event_df = event_df.drop_duplicates(subset=["name", "Timestamp"], keep="last")

    return event_df


def compute_cpu_mem_cost_fast(exp_dir):
    """
    Fast CPU+memory cost calculation.
    Uses the real workload completion time if found.
    """
    docker_log_path = exp_dir / "docker_log.csv"

    if not docker_log_path.exists():
        raise FileNotFoundError(f"Missing file: {docker_log_path}")

    docker_df = load_docker_log(docker_log_path)

    window = find_completion_window(exp_dir, docker_df)

    window_start_ms = window["start_ms"]
    window_end_ms = window["end_ms"]

    event_df = prepare_events_for_window(
        docker_df,
        window_start_ms,
        window_end_ms
    )

    if event_df.empty:
        return {
            "memory_cost": 0.0,
            "cpu_cost": 0.0,
            "cpu_mem_cost": 0.0,
            "memory_time_sec": 0.0,
            "cpu_time_sec": 0.0,
            "duration_sec": (window_end_ms - window_start_ms) / 1000.0,
            "window_source": window["source"],
            "active_containers_at_end": 0,
        }

    # Next status-change timestamp for the same container
    event_df["next_timestamp"] = event_df.groupby("name")["Timestamp"].shift(-1)

    # Last status is charged only until workload completion time
    event_df["next_timestamp"] = event_df["next_timestamp"].fillna(window_end_ms)

    # Clip to experiment window
    event_df["Timestamp"] = event_df["Timestamp"].clip(lower=window_start_ms)
    event_df["next_timestamp"] = event_df["next_timestamp"].clip(upper=window_end_ms)

    event_df["duration_sec"] = (
        event_df["next_timestamp"] - event_df["Timestamp"]
    ) / 1000.0

    event_df["duration_sec"] = event_df["duration_sec"].clip(lower=0)

    memory_statuses = ["running", "paused"]
    cpu_statuses = ["running"]

    memory_time_sec = event_df.loc[
        event_df["status"].isin(memory_statuses),
        "duration_sec"
    ].sum()

    cpu_time_sec = event_df.loc[
        event_df["status"].isin(cpu_statuses),
        "duration_sec"
    ].sum()

    memory_cost = memory_time_sec * MEM_RATE
    cpu_cost = cpu_time_sec * CPU_RATE
    cpu_mem_cost = memory_cost + cpu_cost

    # Debug: how many containers are still active at the end of the window
    last_status = event_df.sort_values(["name", "Timestamp"]).groupby("name").tail(1)
    active_containers_at_end = last_status[
        last_status["status"].isin(memory_statuses)
    ]["name"].nunique()

    return {
        "memory_cost": memory_cost,
        "cpu_cost": cpu_cost,
        "cpu_mem_cost": cpu_mem_cost,
        "memory_time_sec": memory_time_sec,
        "cpu_time_sec": cpu_time_sec,
        "duration_sec": (window_end_ms - window_start_ms) / 1000.0,
        "window_source": window["source"],
        "active_containers_at_end": active_containers_at_end,
    }


# ============================================================
# 5. Run all experiments
# ============================================================

detail_rows = []

for dataset_name, experiments in datasets.items():
    for exp in experiments:
        exp_dir = BASE_DIR / exp["id"]

        result = compute_cpu_mem_cost_fast(exp_dir)

        detail_rows.append({
            "Dataset": dataset_name,
            "Method": exp["name"],
            "Experiment ID": exp["id"],
            "Duration Sec": result["duration_sec"],
            "Memory Time Sec": result["memory_time_sec"],
            "CPU Time Sec": result["cpu_time_sec"],
            "Memory Cost": result["memory_cost"],
            "CPU Cost": result["cpu_cost"],
            "CPU+Mem Cost": result["cpu_mem_cost"],
            "Window Source": result["window_source"],
            "Active Containers at End": result["active_containers_at_end"],
        })

detail_df = pd.DataFrame(detail_rows)

print("\n================ DETAILED CPU + MEMORY COST ================")
print(detail_df.to_string(index=False))

detail_df.to_csv("cpu_mem_cost_detail_fast.csv", index=False)


# ============================================================
# 6. Compare NMIG against each baseline
# ============================================================

comparison_rows = []

for dataset_name in detail_df["Dataset"].unique():
    dataset_df = detail_df[detail_df["Dataset"] == dataset_name].copy()

    nmig_rows = dataset_df[dataset_df["Method"] == "NMIG"]

    if nmig_rows.empty:
        raise ValueError(f"NMIG row not found for dataset: {dataset_name}")

    nmig_cost = nmig_rows["CPU+Mem Cost"].iloc[0]

    for _, row in dataset_df.iterrows():
        if row["Method"] == "NMIG":
            continue

        baseline_cost = row["CPU+Mem Cost"]

        if baseline_cost == 0:
            reduction = np.nan
        else:
            reduction = ((baseline_cost - nmig_cost) / baseline_cost) * 100

        comparison_rows.append({
            "Dataset": dataset_name,
            "Baseline": row["Method"],
            "Baseline CPU+Mem Cost": baseline_cost,
            "NMIG CPU+Mem Cost": nmig_cost,
            "Reduction (%)": reduction,
            "Interpretation": "NMIG better" if reduction >= 0 else "NMIG worse",
        })

comparison_df = pd.DataFrame(comparison_rows)

print("\n================ NMIG VS EACH BASELINE ================")
print(comparison_df.to_string(index=False))

comparison_df.to_csv("cpu_mem_cost_comparison_fast.csv", index=False)


# ============================================================
# 7. Summary range for paper table
# ============================================================

summary_rows = []

for dataset_name in detail_df["Dataset"].unique():
    dataset_comparison = comparison_df[
        comparison_df["Dataset"] == dataset_name
    ].copy()

    min_reduction = dataset_comparison["Reduction (%)"].min()
    max_reduction = dataset_comparison["Reduction (%)"].max()

    nmig_cost = dataset_comparison["NMIG CPU+Mem Cost"].iloc[0]

    summary_rows.append({
        "Dataset": dataset_name,
        "NMIG CPU+Mem Cost": nmig_cost,
        "Min Reduction (%)": min_reduction,
        "Max Reduction (%)": max_reduction,
        "Reduction Range": f"{min_reduction:.2f}--{max_reduction:.2f}%",
    })

summary_df = pd.DataFrame(summary_rows)

print("\n================ CPU + MEMORY REDUCTION SUMMARY ================")
print(summary_df.to_string(index=False))

summary_df.to_csv("cpu_mem_cost_summary_fast.csv", index=False)


# ============================================================
# 8. Print LaTeX-ready rows for your table
# ============================================================

print("\n================ LATEX-READY CPU+MEM VALUES ================")

for _, row in summary_df.iterrows():
    dataset = row["Dataset"]
    cpu_mem_range = row["Reduction Range"]

    print(f"{dataset} & {cpu_mem_range} \\\\")


# ============================================================
# 9. Print warning if any negative values remain
# ============================================================

negative_df = comparison_df[comparison_df["Reduction (%)"] < 0]

print("\n================ NEGATIVE CASES ================")

if negative_df.empty:
    print("No negative cases. NMIG is better than all baselines for CPU+memory cost.")
else:
    print(negative_df.to_string(index=False))
    print("\nWarning: Some baselines still have lower CPU+memory cost than NMIG.")
    print("Check Window Source, Duration Sec, Active Containers at End, and request completion logs.")


================ DETAILED CPU + MEMORY COST ================
Dataset    Method       Experiment ID  Duration Sec  Memory Time Sec  CPU Time Sec  Memory Cost  CPU Cost  CPU+Mem Cost           Window Source  Active Containers at End
 Normal OpenWhisk exp_20250626_205804       15005.0        1103189.0      369203.0  11296.65536  18.46015   11315.11551 docker_log.csv fallback                       134
 Normal      NMIG exp_20250627_042542       14440.0         855595.0       57478.0   8761.29280   2.87390    8764.16670 docker_log.csv fallback                       112
 Normal Histogram exp_20250626_155721       15037.0        4801328.0      479424.0  49165.59872  23.97120   49189.56992 docker_log.csv fallback                       622
 Normal   Pagurus exp_20250626_113035       14977.0         929196.0      252563.0   9514.96704  12.62815    9527.59519 docker_log.csv fallback                       120
Similar OpenWhisk exp_20250625_162425       14994.0         232624.0       63555.0   238

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# 1. Cost parameters
# ============================================================

mem_cost_per_sec_per_mb = 0.00001
cpu_cost_per_sec = 0.00005
memory_mb = 1024

BASE_DIR = Path("../results")


# ============================================================
# 2. Experiments for all datasets
# ============================================================

datasets = {
    "Normal": [
        {"id": "exp_20250626_205804", "name": "OpenWhisk"},
        {"id": "exp_20250627_042542", "name": "NMIG"},
        {"id": "exp_20250626_155721", "name": "Histogram"},
        {"id": "exp_20250626_113035", "name": "Pagurus"},
    ],

    "Similar": [
        {"id": "exp_20250625_162425", "name": "OpenWhisk"},
        {"id": "exp_20250625_113106", "name": "NMIG"},
        {"id": "exp_20250625_211541", "name": "Histogram"},
        {"id": "exp_20250626_050332", "name": "Pagurus"},
    ],

    "Bursty": [
        {"id": "exp_20250625_014444", "name": "OpenWhisk"},
        {"id": "exp_20250625_071954", "name": "NMIG"},
        {"id": "exp_20250624_210136", "name": "Histogram"},
        {"id": "exp_20250624_165438", "name": "Pagurus"},
    ],
}


# ============================================================
# 3. Your original cost-calculation method
# ============================================================

def compute_cpu_mem_cost_using_original_method(exp_id):
    path = BASE_DIR / exp_id / "docker_log.csv"

    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    # Same as your logic, but without reading the file twice
    df = pd.read_csv(
        path,
        names=["Timestamp", "name", "image", "status"]
    )

    # Basic cleaning
    df["Timestamp"] = pd.to_numeric(df["Timestamp"], errors="coerce")
    df = df.dropna(subset=["Timestamp", "name", "status"])
    df["Timestamp"] = df["Timestamp"].astype("int64")

    df["name"] = df["name"].astype(str).str.strip()
    df["status"] = df["status"].astype(str).str.strip().str.lower()

    start_time = df["Timestamp"].min()
    df["ElapsedTime"] = (df["Timestamp"] - start_time) / 1000.0

    # ------------------------------------------------------------
    # Your original logic starts here
    # ------------------------------------------------------------

    # Process timestamps
    timestamps = sorted(df["Timestamp"].unique())

    # If there are not enough timestamps, return zero cost
    if len(timestamps) < 2:
        return {
            "elapsed_secs": [],
            "cumulative_costs": [],
            "final_cost": 0.0,
            "duration_sec": 0.0,
            "num_timestamps": len(timestamps),
            "num_containers": df["name"].nunique(),
        }

    # Build container status per timestamp
    container_status = {}

    for name, group in df.groupby("name"):
        group = group.sort_values("Timestamp")
        status_list = []
        last_status = None
        idx = 0

        for t in timestamps:
            while idx < len(group) and group.iloc[idx]["Timestamp"] <= t:
                last_status = group.iloc[idx]["status"]
                idx += 1

            status_list.append(last_status)

        container_status[name] = status_list

    # Calculate cumulative cost
    cumulative_costs = []
    total_cost = 0.0

    for i in range(1, len(timestamps)):
        interval_sec = (timestamps[i] - timestamps[i - 1]) / 1000.0

        for name, status_list in container_status.items():
            status = status_list[i - 1]

            if status is None:
                continue

            mem_cost = interval_sec * mem_cost_per_sec_per_mb * memory_mb

            if status == "running":
                cpu_cost = interval_sec * cpu_cost_per_sec
                total_cost += mem_cost + cpu_cost

            elif status == "paused":
                total_cost += mem_cost

        cumulative_costs.append(total_cost)

    elapsed_secs = [(t - timestamps[0]) / 1000.0 for t in timestamps[1:]]

    # ------------------------------------------------------------
    # Your original logic ends here
    # ------------------------------------------------------------

    return {
        "elapsed_secs": elapsed_secs,
        "cumulative_costs": cumulative_costs,
        "final_cost": cumulative_costs[-1] if cumulative_costs else 0.0,
        "duration_sec": elapsed_secs[-1] if elapsed_secs else 0.0,
        "num_timestamps": len(timestamps),
        "num_containers": df["name"].nunique(),
    }


# ============================================================
# 4. Run all experiments using your original method
# ============================================================

plot_data = {}
detail_rows = []

for dataset_name, experiments in datasets.items():
    plot_data[dataset_name] = {}

    for exp in experiments:
        result = compute_cpu_mem_cost_using_original_method(exp["id"])

        # Store plotting data
        plot_data[dataset_name][exp["name"]] = (
            result["elapsed_secs"],
            result["cumulative_costs"]
        )

        # Store final cost
        detail_rows.append({
            "Dataset": dataset_name,
            "Method": exp["name"],
            "Experiment ID": exp["id"],
            "CPU+Mem Cost": result["final_cost"],
            "Duration Sec": result["duration_sec"],
            "Timestamps": result["num_timestamps"],
            "Containers": result["num_containers"],
        })

detail_df = pd.DataFrame(detail_rows)

print("\n================ DETAILED CPU + MEMORY COST ================")
print(detail_df.to_string(index=False))

detail_df.to_csv("cpu_mem_cost_detail_original_method.csv", index=False)


# ============================================================
# 5. Compare NMIG against each baseline
# ============================================================

comparison_rows = []

for dataset_name in detail_df["Dataset"].unique():
    dataset_df = detail_df[detail_df["Dataset"] == dataset_name].copy()

    nmig_row = dataset_df[dataset_df["Method"] == "NMIG"]

    if nmig_row.empty:
        raise ValueError(f"NMIG result not found for dataset: {dataset_name}")

    nmig_cost = nmig_row["CPU+Mem Cost"].iloc[0]

    for _, row in dataset_df.iterrows():
        if row["Method"] == "NMIG":
            continue

        baseline_cost = row["CPU+Mem Cost"]

        if baseline_cost == 0:
            reduction = np.nan
        else:
            reduction = ((baseline_cost - nmig_cost) / baseline_cost) * 100

        comparison_rows.append({
            "Dataset": dataset_name,
            "Baseline": row["Method"],
            "Baseline CPU+Mem Cost": baseline_cost,
            "NMIG CPU+Mem Cost": nmig_cost,
            "Reduction (%)": reduction,
            "Interpretation": "NMIG better" if reduction >= 0 else "NMIG worse",
        })

comparison_df = pd.DataFrame(comparison_rows)

print("\n================ NMIG VS EACH BASELINE ================")
print(comparison_df.to_string(index=False))

comparison_df.to_csv("cpu_mem_cost_comparison_original_method.csv", index=False)


# ============================================================
# 6. Show exact cases where NMIG is worse
# ============================================================

negative_df = comparison_df[comparison_df["Reduction (%)"] < 0]

print("\n================ EXACT CASES WHERE NMIG IS WORSE ================")

if negative_df.empty:
    print("NMIG is not worse than any baseline for CPU+memory cost.")
else:
    for _, row in negative_df.iterrows():
        print(
            f"In the {row['Dataset']} dataset, NMIG is worse than {row['Baseline']} "
            f"for CPU+memory cost. "
            f"Baseline cost = {row['Baseline CPU+Mem Cost']:.5f}, "
            f"NMIG cost = {row['NMIG CPU+Mem Cost']:.5f}, "
            f"reduction = {row['Reduction (%)']:.2f}%."
        )

negative_df.to_csv("cpu_mem_cost_negative_cases_original_method.csv", index=False)


# ============================================================
# 7. Summary reduction range for paper table
# ============================================================

summary_rows = []

for dataset_name in comparison_df["Dataset"].unique():
    dataset_comparison = comparison_df[
        comparison_df["Dataset"] == dataset_name
    ]

    min_reduction = dataset_comparison["Reduction (%)"].min()
    max_reduction = dataset_comparison["Reduction (%)"].max()

    nmig_cost = dataset_comparison["NMIG CPU+Mem Cost"].iloc[0]

    summary_rows.append({
        "Dataset": dataset_name,
        "NMIG CPU+Mem Cost": nmig_cost,
        "Min Reduction (%)": min_reduction,
        "Max Reduction (%)": max_reduction,
        "Reduction Range": f"{min_reduction:.2f}--{max_reduction:.2f}%",
    })

summary_df = pd.DataFrame(summary_rows)

print("\n================ CPU + MEMORY REDUCTION SUMMARY ================")
print(summary_df.to_string(index=False))

summary_df.to_csv("cpu_mem_cost_summary_original_method.csv", index=False)


# ============================================================
# 8. Print LaTeX-ready values for your table
# ============================================================

print("\n================ LATEX-READY CPU+MEM VALUES ================")

for _, row in summary_df.iterrows():
    print(f"{row['Dataset']} & {row['Reduction Range']} \\\\")


# ============================================================
# 9. Optional: print final table rows with your verified GPU values
# ============================================================

gpu_reduction = {
    "Normal": "98.58--98.68\\%",
    "Similar": "99.12--99.25\\%",
    "Bursty": "99.47--99.50\\%",
}

print("\n================ LATEX-READY TABLE ROWS ================")

for _, row in summary_df.iterrows():
    dataset = row["Dataset"]
    cpu_mem_red = row["Reduction Range"].replace("%", "\\%")
    gpu_red = gpu_reduction[dataset]

    print(
        f"{dataset} & {gpu_red} & {cpu_mem_red} "
        f"& xxxx & x & xx.xs & xx.xs \\\\"
    )


================ DETAILED CPU + MEMORY COST ================
Dataset    Method       Experiment ID  CPU+Mem Cost  Duration Sec  Timestamps  Containers
 Normal OpenWhisk exp_20250626_205804  11315.115510       15005.0       14260         153
 Normal      NMIG exp_20250627_042542   8764.166700       14440.0       13564         128
 Normal Histogram exp_20250626_155721  49189.569922       15037.0       13635         744
 Normal   Pagurus exp_20250626_113035   9527.595190       14977.0       14264         142
Similar OpenWhisk exp_20250625_162425   2385.247510       14994.0       14496          40
Similar      NMIG exp_20250625_113106   3352.360100       14445.0       13677          37
Similar Histogram exp_20250625_211541  37137.602700       14974.0       13925         597
Similar   Pagurus exp_20250626_050332   2782.512730       14994.0       14495          42
 Bursty OpenWhisk exp_20250625_014444  13707.115090       14223.0        9633         167
 Bursty      NMIG exp_20250625_071954 

In [10]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path("../results")
memory_mb = 1024
memory_gb = memory_mb / 1024.0

datasets = {
    "Normal": [
        {"id": "exp_20250626_205804", "name": "OpenWhisk"},
        {"id": "exp_20250627_042542", "name": "NMIG"},
        {"id": "exp_20250626_155721", "name": "Histogram"},
        {"id": "exp_20250626_113035", "name": "Pagurus"},
    ],
    "Similar": [
        {"id": "exp_20250625_162425", "name": "OpenWhisk"},
        {"id": "exp_20250625_113106", "name": "NMIG"},
        {"id": "exp_20250625_211541", "name": "Histogram"},
        {"id": "exp_20250626_050332", "name": "Pagurus"},
    ],
    "Bursty": [
        {"id": "exp_20250625_014444", "name": "OpenWhisk"},
        {"id": "exp_20250625_071954", "name": "NMIG"},
        {"id": "exp_20250624_210136", "name": "Histogram"},
        {"id": "exp_20250624_165438", "name": "Pagurus"},
    ],
}


def compute_cpu_memory_separately(exp_id):
    path = BASE_DIR / exp_id / "docker_log.csv"

    df = pd.read_csv(
        path,
        names=["Timestamp", "name", "image", "status"]
    )

    df["Timestamp"] = pd.to_numeric(df["Timestamp"], errors="coerce")
    df = df.dropna(subset=["Timestamp", "name", "status"])
    df["Timestamp"] = df["Timestamp"].astype("int64")

    df["name"] = df["name"].astype(str).str.strip()
    df["status"] = df["status"].astype(str).str.strip().str.lower()

    timestamps = sorted(df["Timestamp"].unique())

    if len(timestamps) < 2:
        return {
            "cpu_active_sec": 0.0,
            "memory_residency_gb_sec": 0.0,
            "duration_sec": 0.0,
            "containers": df["name"].nunique(),
        }

    container_status = {}

    for name, group in df.groupby("name"):
        group = group.sort_values("Timestamp")
        status_list = []
        last_status = None
        idx = 0

        for t in timestamps:
            while idx < len(group) and group.iloc[idx]["Timestamp"] <= t:
                last_status = group.iloc[idx]["status"]
                idx += 1

            status_list.append(last_status)

        container_status[name] = status_list

    cpu_active_sec = 0.0
    memory_residency_gb_sec = 0.0

    for i in range(1, len(timestamps)):
        interval_sec = (timestamps[i] - timestamps[i - 1]) / 1000.0

        for name, status_list in container_status.items():
            status = status_list[i - 1]

            if status is None:
                continue

            if status == "running":
                cpu_active_sec += interval_sec
                memory_residency_gb_sec += interval_sec * memory_gb

            elif status == "paused":
                memory_residency_gb_sec += interval_sec * memory_gb

    duration_sec = (timestamps[-1] - timestamps[0]) / 1000.0

    return {
        "cpu_active_sec": cpu_active_sec,
        "memory_residency_gb_sec": memory_residency_gb_sec,
        "duration_sec": duration_sec,
        "containers": df["name"].nunique(),
    }


# ============================================================
# Run all experiments
# ============================================================

detail_rows = []

for dataset_name, experiments in datasets.items():
    for exp in experiments:
        result = compute_cpu_memory_separately(exp["id"])

        detail_rows.append({
            "Dataset": dataset_name,
            "Method": exp["name"],
            "Experiment ID": exp["id"],
            "CPU Active Time Sec": result["cpu_active_sec"],
            "Memory Residency GB-sec": result["memory_residency_gb_sec"],
            "Duration Sec": result["duration_sec"],
            "Containers": result["containers"],
        })

detail_df = pd.DataFrame(detail_rows)

print("\n================ SEPARATE CPU AND MEMORY USAGE ================")
print(detail_df.to_string(index=False))

detail_df.to_csv("separate_cpu_memory_usage.csv", index=False)


# ============================================================
# Compare NMIG against each baseline separately
# ============================================================

comparison_rows = []

for dataset_name in detail_df["Dataset"].unique():
    dataset_df = detail_df[detail_df["Dataset"] == dataset_name].copy()

    nmig_row = dataset_df[dataset_df["Method"] == "NMIG"]

    if nmig_row.empty:
        raise ValueError(f"NMIG result not found for dataset: {dataset_name}")

    nmig_cpu = nmig_row["CPU Active Time Sec"].iloc[0]
    nmig_mem = nmig_row["Memory Residency GB-sec"].iloc[0]

    for _, row in dataset_df.iterrows():
        if row["Method"] == "NMIG":
            continue

        baseline_cpu = row["CPU Active Time Sec"]
        baseline_mem = row["Memory Residency GB-sec"]

        cpu_reduction = ((baseline_cpu - nmig_cpu) / baseline_cpu) * 100 if baseline_cpu != 0 else np.nan
        mem_reduction = ((baseline_mem - nmig_mem) / baseline_mem) * 100 if baseline_mem != 0 else np.nan

        comparison_rows.append({
            "Dataset": dataset_name,
            "Baseline": row["Method"],
            "Baseline CPU Active Time Sec": baseline_cpu,
            "NMIG CPU Active Time Sec": nmig_cpu,
            "CPU Active Time Reduction (%)": cpu_reduction,
            "Baseline Memory Residency GB-sec": baseline_mem,
            "NMIG Memory Residency GB-sec": nmig_mem,
            "Memory Residency Reduction (%)": mem_reduction,
        })

comparison_df = pd.DataFrame(comparison_rows)

print("\n================ NMIG VS BASELINES, CPU AND MEMORY SEPARATED ================")
print(comparison_df.to_string(index=False))

comparison_df.to_csv("separate_cpu_memory_comparison.csv", index=False)


# ============================================================
# Summary range for paper table
# ============================================================

summary_rows = []

for dataset_name in comparison_df["Dataset"].unique():
    dataset_comp = comparison_df[comparison_df["Dataset"] == dataset_name]

    cpu_min = dataset_comp["CPU Active Time Reduction (%)"].min()
    cpu_max = dataset_comp["CPU Active Time Reduction (%)"].max()

    mem_min = dataset_comp["Memory Residency Reduction (%)"].min()
    mem_max = dataset_comp["Memory Residency Reduction (%)"].max()

    summary_rows.append({
        "Dataset": dataset_name,
        "CPU Active Time Reduction Range": f"{cpu_min:.2f}--{cpu_max:.2f}%",
        "Memory Residency Reduction Range": f"{mem_min:.2f}--{mem_max:.2f}%",
    })

summary_df = pd.DataFrame(summary_rows)

print("\n================ SUMMARY FOR PAPER TABLE ================")
print(summary_df.to_string(index=False))

summary_df.to_csv("separate_cpu_memory_summary.csv", index=False)


# ============================================================
# LaTeX-ready rows
# ============================================================

print("\n================ LATEX-READY ROWS ================")

gpu_reduction = {
    "Normal": "98.58--98.68\\%",
    "Similar": "99.12--99.25\\%",
    "Bursty": "99.47--99.50\\%",
}

for _, row in summary_df.iterrows():
    dataset = row["Dataset"]

    cpu_range = row["CPU Active Time Reduction Range"].replace("%", "\\%")
    mem_range = row["Memory Residency Reduction Range"].replace("%", "\\%")

    print(
        f"{dataset} & {gpu_reduction[dataset]} & {cpu_range} & {mem_range} "
        f"& xxxx & x & xx.xs & xx.xs \\\\"
    )


================ SEPARATE CPU AND MEMORY USAGE ================
Dataset    Method       Experiment ID  CPU Active Time Sec  Memory Residency GB-sec  Duration Sec  Containers
 Normal OpenWhisk exp_20250626_205804             369203.0                1103189.0       15005.0         153
 Normal      NMIG exp_20250627_042542              57478.0                 855595.0       14440.0         128
 Normal Histogram exp_20250626_155721             479424.0                4801328.0       15037.0         744
 Normal   Pagurus exp_20250626_113035             252563.0                 929196.0       14977.0         142
Similar OpenWhisk exp_20250625_162425              63555.0                 232624.0       14994.0          40
Similar      NMIG exp_20250625_113106              89890.0                 326940.0       14445.0          37
Similar Histogram exp_20250625_211541             531254.0                3624125.0       14974.0         597
Similar   Pagurus exp_20250626_050332              6814